In [1]:
# xgboost_dask_refactor.py
"""Train an XGBoost model on hydro‑meteorological data using Dask.

This refactor focuses on
1. **Reproducibility** – fixed random seeds, deterministic sampling.
2. **Clarity & Modularity** – small, typed functions with docstrings.
3. **Efficiency** – single‑pass lag generation, limited copies, eager clean‑up.
4. **Robustness** – explicit async handling, context‑managed client, safe path handling.
"""
from __future__ import annotations

In [2]:
import asyncio
from pathlib import Path
from typing import Iterable, List

In [3]:
import dask.array as da
import dask.dataframe as dd
import numpy as np
import xgboost as xgb
from dask.distributed import Client, LocalCluster

In [4]:
from utils.datasets import (
    ARTIFACTS_FOLDER,
    ROOT_FOLDER,
    TimeRange,
)

ImportError: cannot import name 'TimeRange' from 'utils.datasets' (/home/khuzin/Projects/2025-m1p-private/code/src/utils/datasets/__init__.py)

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# Configuration
# ────────────────────────────────────────────────────────────────────────────────
SEED = 42
PATH_MERGED_DATASETS = ARTIFACTS_FOLDER / "merged_datasets"
FILENAME_TRAIN_IDS = "train_file_ids.csv"

In [5]:
TARGET = "q_mm_day"
LAG_FEATURES: List[str] = [
    "prcp",
    "t_max",
    "t_min",
    "t_mean",
    TARGET,
    "lvl_sm",
]
HORIZON_HISTORY = TimeRange.YEAR  # 365
HORIZON_FORECAST = TimeRange.WEEK  # 7

NameError: name 'TimeRange' is not defined

In [ ]:
# XGBoost hyper‑parameters -------------------------------------------------------
XGB_PARAMS = {
    "tree_method": "hist",
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "seed": SEED,
}

In [ ]:
EPS = 1e-6  # smoothing to avoid /0 in MAPE

────────────────────────────────────────────────────────────────────────────────
Data helpers
────────────────────────────────────────────────────────────────────────────────

In [ ]:
def sample_file_ids(n: int, seed: int = SEED) -> List[int]:
    """Randomly sample *n* file IDs (with replacement=False)."""
    import polars as pl  # local import keeps global namespace clean

    ids = (
        pl.read_csv(PATH_MERGED_DATASETS / FILENAME_TRAIN_IDS)
        .sample(n, seed=seed)
        .get_column("file_id")
        .to_list()
    )
    return ids

In [ ]:
def load_parquet_subset(ids: Iterable[int]) -> dd.DataFrame:
    paths = [PATH_MERGED_DATASETS / f"{fid}.parquet" for fid in ids]
    return dd.read_parquet(paths).set_index("gauge_id")

In [ ]:
def add_lags(df: dd.DataFrame, columns: Iterable[str], max_lag: int) -> dd.DataFrame:
    """Add lagged versions of *columns* up to *max_lag* inclusive."""

    def _lag_partition(pdf):
        pdf = pdf.sort_values("date")
        for col in columns:
            for lag in range(1, max_lag + 1):
                pdf[f"{col}_lag{lag}"] = pdf[col].shift(lag)
        return pdf

    return df.map_partitions(_lag_partition, meta=df)

────────────────────────────────────────────────────────────────────────────────
XGBoost – custom objective & metric
────────────────────────────────────────────────────────────────────────────────

In [ ]:
def mape_obj(preds: np.ndarray, dtrain: xgb.DMatrix):
    y = dtrain.get_label()
    grad = np.sign(preds - y) / (np.abs(y) + EPS)
    hess = EPS / (np.abs(y) + EPS) ** 2
    return grad, hess

In [ ]:
def mape_eval(preds: np.ndarray, dtrain: xgb.DMatrix):
    y = dtrain.get_label()
    mape = np.mean(np.abs((y - preds) / (y + EPS))) * 100
    return "MAPE", mape, False  # False ⇒ lower is better

────────────────────────────────────────────────────────────────────────────────
Training pipeline
────────────────────────────────────────────────────────────────────────────────

In [ ]:
async def train_async():
    # Cluster ---------------------------------------------------------------
    cluster = LocalCluster(
        n_workers=4,
        threads_per_worker=2,
        memory_limit="4GB",
        asynchronous=True,
    )
    async with Client(cluster, asynchronous=True) as client:  # dashboard :8787
        # Data ----------------------------------------------------------
        ids = sample_file_ids(10)
        df = load_parquet_subset(ids)
        df = add_lags(df, LAG_FEATURES, HORIZON_HISTORY)
        df = df.dropna()

        X = df.drop(columns=[TARGET])
        y = df[TARGET]

        dtrain = await xgb.dask.DaskDMatrix(client, X, y)

        booster = await xgb.dask.train(
            client,
            XGB_PARAMS,
            dtrain,
            num_boost_round=500,
            obj=mape_obj,
            evals=[(dtrain, "train")],
            feval=mape_eval,
        )

        # Persist model
        model_path = ROOT_FOLDER / "models" / "xgb_mape.json"
        model_path.parent.mkdir(parents=True, exist_ok=True)
        booster["booster"].save_model(model_path.as_posix())
        print(f"Model saved to {model_path.relative_to(ROOT_FOLDER)}")

In [ ]:
def main() -> None:
    asyncio.run(train_async())

In [ ]:
if __name__ == "__main__":
    main()